In [1]:
from automated_events_coding.pipeline import (
    step0_relevance,
    step1_events,
    step2_location,
    step3_event_lead,
    step4_interaction,
    step5_identification,
)
from automated_events_coding.pipeline.schemas import Event, EventList, EventLead, EventRecord, InteractionType
from automated_events_coding.llm.client import get_client
from automated_events_coding.llm.server import LlamaServerConfig, LlamaServer

In [2]:
article_text = """
First shots fired in water warCalueque water no longer free It appears that Namibia has been reneging on a deal with Angola under which the northern regions are receiving free water from Angola's Calueque Dam. Delays by the Namibian authorities to establish a conveyance system that would supply sufficient potable water from the Oshakati purification plant to the Cunene Province in Angola under the Kunene Transboundary Water Supply Project (KTWSP) agreement are likely to cost Namibia. 

The agreement was aimed at strengthening the 1964 Cunene River Scheme Agreement under which Namibia would supply people in southern Angola with purified water sourced from the Calueque Dam. 

It is further reported that due to this delay, and a need to have potable water supplied to southern Angola, the Angolan government established the Xangongo Water Supply Scheme to provide this water to its people. Now the Angolan authorities want Namibia to start paying for the water it gets from the Calueque Dam. 

Unconfirmed reports indicate the figure is in the region of millions of dollars a month. 

It is reported that under the KTWSP agreement that was supposed to be implemented between May 2010 and June 2012, Namibia was supposed to supply potable water to Angola's Cunene Province from Oshakati, where water from Calueque is treated. 

Namibia was expected to upgrade its purification plant at Oshakati and expand the existing bulk supply system at Omakango, Omafo and Oshikango. 

The Angolan established a water body for the Cunene Province (Empresa de Água e Saneamento da Província do Cunene, EASC-EP) to maintain water supply and wastewater schemes in the province. 

As far back as 2005 the Angolan government installed a 40-kilometre pipeline between the border town of Santa Clara (across the border from Oshikango) and Ondjiva and they were ready to receive water from Namibia.  

After years of delays by the Namibian government, the Angolan government went on to establish a water supply scheme, Xangongo, to supply water to Cunene Province. 

The Angolans are now abstracting water from the Kunene River at Xangongo, 75km upstream from Calueque, treating it and then distributing to towns and villages including Ondjiva, Namacunde, Chiede and Santa Clara.  

This included the construction of a water treatment plant with a total capacity of about 40 000 cubic metres per day at Xangongo and the construction of a transmission pipe network from Xangongo via Ondjiva to Santa Clara and Chiede.

In terms of the agreement, NamPower is responsible for supplying power to Calueque and Xangongo via a power line from the Ruacana hydropower station. 

Currently, the northern regions are facing serious water restrictions until the end of February. The water supply is cut off from 22:00 to 05:00 every day. 

According to NamWater, the supply is interrupted because of rehabilitation work at Angola's Calueque Dam. 

The state of the infrastructure at Calueque has not been established, as journalists have not been permitted to visit the dam. 

However, there have been reports that the work was finished but there is no power to pump the water. Earlier, NamWater said the upgraded facilities were not yet in use as they didn't have electricity. The old facilities were being used in the meantime, the utility said.  

NamPower rejected claims that it had failed to supply power to the Calueque Dam as "factually incorrect and misleading". It said it was NamWater that had to provide a temporary power supply to the new pump station control room. 

NamPower spokesperson Rosa Nikanor said there had been extensive engagements between NamPower, NamWater, RNT (the national transmission utility of Angola), and GABHIC, (the Angolan government entity that is managing the Kunene River Basin and currently operating Calueque Scheme), on possible solutions for power supply to the dam. 

"In order to ensure optimal operation of the dam's sluice gates in the interim, NamWater will provide a temporary power supply to the new pump station control room.  

"NamPower is also investigating a solution which involves upgrading and strengthening the existing power supply infrastructure (in Namibia).  

"This will ensure reliable operation of the Calueque Scheme.  

"The long-term power supply solution will include the construction of a 132kV power line from the Hippo substation at Ruacana to a new substation (30/6.6kV) at the Calueque Dam," Nikanor said. 

The Ministry of Agriculture, Water and Forestry refused to comment, saying NamWater was in better position to comment. 

ILENI NANDJATO 
http://imgs.syndigate.info/637/1808/22/151667654322.jpg

Load-Date: January 23, 2018
""".strip()

In [3]:
relevance = step0_relevance.check_relevance(article_text=article_text)
relevance

config.json:   0%|          | 0.00/2.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/379 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.58GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

RelevanceResult(relevant=True, meta={'relevance_label': 'Y', 'relevance_score': '0.8526'})

In [ ]:
from automated_events_coding.pipeline.prompts import STEP1_EVENT_EXTRACTION
import re
import json
from pydantic import TypeAdapter

json_pattern = r"\[.*?\]"
llama_server_config = LlamaServerConfig(
    model_path="/Users/pnadel01/models/gguf/unsloth_Qwen3.8-27B-GGUF/Qwen3.8-27B-UD-Q5_K_XL.gguf",
    temperature=0.7,
    top_p=0.80,
    top_k=20,
    extra_args=["--reasoning", "0"]
)

with LlamaServer(llama_server_config) as server:
    client = get_client(llama_server_config)

    # step 1
    response = client.chat.completions.create(
        model="Test",
        messages=[
            {"role": "system", "content": STEP1_EVENT_EXTRACTION},
            {"role": "user", "content": article_text},
        ],
    )
    raw = response.choices[0].message.content
    content = json.loads(re.findall(json_pattern, raw, re.DOTALL)[0])
    events = []
    for event in content:
        event_obj = Event(text=event["event_summary"])
        events.append(event_obj)
    validated = EventList(events=events)
    print(validated)

events=[Event(text='Angolan authorities want Namibia to start paying for the water it gets from the Calueque Dam.'), Event(text='The Angolan government established the Xangongo Water Supply Scheme to provide water to its people.'), Event(text='The Angolans are now abstracting water from the Kunene River at Xangongo, treating it and then distributing to towns and villages including Ondjiva, Namacunde, Chiede and Santa Clara.'), Event(text='The northern regions are facing serious water restrictions until the end of February.'), Event(text='The water supply is cut off from 22:00 to 05:00 every day.'), Event(text="NamPower rejected claims that it had failed to supply power to the Calueque Dam as 'factually incorrect and misleading'."), Event(text='NamPower is investigating a solution which involves upgrading and strengthening the existing power supply infrastructure.'), Event(text='The Ministry of Agriculture, Water and Forestry refused to comment.')]


In [ ]:
from pprint import pprint
for i, event in enumerate(validated.events):
    print(f"Event {i+1}")
    pprint(event.text)
    print("-"*40)

Event 1
('Namibia and Angola enter into a dispute over water payments for the Calueque '
 'Dam, as Angola demands payment for water previously provided for free due to '
 "delays in Namibia's Kunene Transboundary Water Supply Project.")
----------------------------------------
Event 2
('The Angolan government establishes the Xangongo Water Supply Scheme to '
 'provide potable water to Cunene Province after years of delays by Namibian '
 'authorities in implementing the KTWSP agreement.')
----------------------------------------
Event 3
('Namibia faces water restrictions in northern regions, with supply cut off '
 'daily from 22:00 to 05:00, attributed by NamWater to rehabilitation work at '
 "Angola's Calueque Dam.")
----------------------------------------
Event 4
('NamPower and NamWater engage in disputes and negotiations regarding power '
 'supply to the Calueque Dam, with NamPower rejecting claims of failure to '
 'supply power and NamWater providing temporary power for pump statio